## Student Attendance Analytics with Pandas

In [1]:
import pandas as pd
import numpy as np

#### Section 0: Data Loading

In [3]:
df = pd.read_excel("csec_attendance.xlsx", sheet_name="Student Summary")
df

,Roll No,Student Name,Subject,Total Classes,Present,Absent,Excused,Attendance %,Status,Classes Needed (75%),Last Present Date,Consecutive Absences
0,4MW24CS123,SAKSHI SHETTY,Data Science & GenAI - 3nd Yr - CSE - C,12,12,0,0,100%,Safe,✓ Met,2026-10-09,0
1,4MW24CS124,SAMEEKSHA BEKAL,Data Science & GenAI - 3nd Yr - CSE - C,12,11,1,0,92%,Safe,✓ Met,2026-10-07,1
2,4MW24CS125,SAMEER ACHARYA,Data Science & GenAI - 3nd Yr - CSE - C,12,2,10,0,17%,Critical,28 more,2026-09-25,3
3,4MW24CS126,SAMHITA S RAO,Data Science & GenAI - 3nd Yr - CSE - C,12,9,3,0,75%,Warning,✓ Met,2026-10-09,0
4,4MW24CS127,SAMIKSHA,Data Science & GenAI - 3nd Yr - CSE - C,12,10,2,0,83%,Warning,✓ Met,2026-10-07,1
...,...,...,...,...,...,...,...,...,...,...,...,...
60,4MW24CS900,RIYA SHETTY,Data Science & GenAI - 3nd Yr - CSE - C,12,7,5,0,58%,Critical,8 more,2026-10-09,0
61,4MW25CS401,ANIRUDH,Data Science & GenAI - 3nd Yr - CSE - C,12,8,4,0,67%,Critical,4 more,2026-10-07,1
62,4MW25CS407,SANATH S DEVADIGA,Data Science & GenAI - 3nd Yr - CSE - C,12,1,11,0,8%,Critical,32 more,2026-09-04,10
63,4MW25CS408,SUSHANTH,Data Science & GenAI - 3nd Yr - CSE - C,12,1,11,0,8%,Critical,32 more,2026-09-04,10


#### Section 1: Data Inspection

In [ ]:
"""Q1. Display the first 10 rows, the shape of the DataFrame, and the data types of each column."""
#-----------------------------Write Solution Code-----------------------------
display(df.head(10))
print("Shape:", df.shape)
print()
print(df.dtypes)

NameError: name 'df' is not defined

In [ ]:
"""Q2. Check for missing values in each column. Identify which columns have null values (especially Last Present Date with "Never") and report the count. """
#-----------------------------Write Solution Code-----------------------------
# Real NaN values in every column
print(df.isnull().sum())

# "Never" is stored as a text value, so isnull() does not catch it
never_count = (df["Last Present Date"] == "Never").sum()
print("\nStudents with Last Present Date = 'Never':", never_count)

In [ ]:
"""Q3. Rename the columns to snake_case (e.g., Roll No → roll_no, Student Name → student_name, Attendance % → attendance_pct). Display the updated column list."""
#-----------------------------Write Solution Code-----------------------------
df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(r"[^a-z0-9]+", "_", regex=True)   # spaces, %, ( ) -> _
      .str.strip("_")
)
# Attendance % -> attendance, so rename it to the requested name
df = df.rename(columns={"attendance": "attendance_pct"})
print(list(df.columns))

In [ ]:
"""Q4. Convert Attendance % from string format (e.g., "64%") to a numeric float column (0.64 or 64.0). Also convert Last Present Date to datetime, handling "Never" as NaT."""
#-----------------------------Write Solution Code-----------------------------
# "64%" -> 64.0
df["attendance_pct"] = df["attendance_pct"].astype(str).str.replace("%", "", regex=False).astype(float)

# "Never" -> NaT
df["last_present_date"] = pd.to_datetime(df["last_present_date"], errors="coerce")

print(df[["attendance_pct", "last_present_date"]].dtypes)
print("NaT count:", df["last_present_date"].isna().sum())

#### Section 2: Filtering & Selection

In [ ]:
"""Q5. Filter and display all students whose Status is "Critical". Sort them by Attendance % in ascending order."""
#-----------------------------Write Solution Code-----------------------------
critical = df[df["status"] == "Critical"].sort_values("attendance_pct", ascending=True)
critical

In [ ]:
"""Q6. Extract all students who have 0 present days (fully absent). List their roll numbers, names, and consecutive absences."""
#-----------------------------Write Solution Code-----------------------------
df[df["present"] == 0][["roll_no", "student_name", "consecutive_absences"]]

In [ ]:
"""Q7. Using .loc and .iloc, display:
(a) All "Safe" students using .loc
(b) The first 15 rows and columns 2-6 using .iloc """
#-----------------------------Write Solution Code-----------------------------
# (a) All "Safe" students using .loc
safe_students = df.loc[df["status"] == "Safe"]
display(safe_students)

# (b) First 15 rows and columns 2-6 (positions 2 to 6) using .iloc
df.iloc[:15, 2:7]

In [ ]:
"""Q8. Find all students whose Consecutive Absences is greater than or equal to 5. Display their name, status, and last present date."""
#-----------------------------Write Solution Code-----------------------------
df.loc[df["consecutive_absences"] >= 5, ["student_name", "status", "last_present_date"]]

### Section 3: Grouping & Aggregation 

In [ ]:
"""Q9. Group the data by Status and compute: 
a) Count of students 
b) Average attendance %  
c) Maximum consecutive absences"""
#-----------------------------Write Solution Code-----------------------------
status_summary = df.groupby("status").agg(
    student_count=("roll_no", "count"),
    avg_attendance_pct=("attendance_pct", "mean"),
    max_consecutive_absences=("consecutive_absences", "max"),
)
status_summary

In [ ]:
"""Q10. Create a new column Attendance Category:
≥ 90% → "Excellent"
75-89% → "Good"
60-74% → "Average"
< 60% → "Poor"
Then use groupby() to count students in each category."""
#-----------------------------Write Solution Code-----------------------------
df["attendance_category"] = np.select(
    [df["attendance_pct"] >= 90,
     df["attendance_pct"] >= 75,
     df["attendance_pct"] >= 60],
    ["Excellent", "Good", "Average"],
    default="Poor",
)

df.groupby("attendance_category")["roll_no"].count().reindex(["Excellent", "Good", "Average", "Poor"])

In [ ]:
"""Q11. Calculate the total number of classes conducted (sum of Total Classes) and the total present/absent counts across all students using .agg()."""
#-----------------------------Write Solution Code-----------------------------
totals = df[["total_classes", "present", "absent"]].agg("sum")
print(totals)
print("\nOverall attendance %:", round(totals["present"] / totals["total_classes"] * 100, 2))

In [ ]:
"""Q12. Group by Status and use .agg() to find:
a)min, max, and mean of Present
b)sum of Absent"""
#-----------------------------Write Solution Code-----------------------------
df.groupby("status").agg(
    present_min=("present", "min"),
    present_max=("present", "max"),
    present_mean=("present", "mean"),
    absent_sum=("absent", "sum"),
)

### Section 4: Feature Engineering & Transformation

In [ ]:
"""Q13. Create a new column Risk Score using the formula:
Risk Score = (Absent * 0.4) + (Consecutive Absences * 0.6)
Display the top 10 students with the highest risk score."""
#-----------------------------Write Solution Code-----------------------------
df["risk_score"] = (df["absent"] * 0.4) + (df["consecutive_absences"] * 0.6)
top10_risk = df.nlargest(10, "risk_score")[["roll_no", "student_name", "absent", "consecutive_absences", "risk_score"]]
top10_risk

In [ ]:
"""Q14. Add a column Classes Needed (Numeric) by extracting the numeric value from the Classes Needed (75%) column (e.g., "5 more" → 5, "✓ Met" → 0)."""
#-----------------------------Write Solution Code-----------------------------
# "28 more" -> 28, "✓ Met" -> no digits -> 0
df["classes_needed_numeric"] = (
    df["classes_needed_75"].str.extract(r"(\d+)")[0].fillna(0).astype(int)
)
df[["student_name", "classes_needed_75", "classes_needed_numeric"]].head(15)

In [ ]:
"""Q15. Using apply() with a lambda function, create a column Attendance Flag that returns:
a)"Defaulter" if attendance < 75%
b)"Safe" otherwise """
#-----------------------------Write Solution Code-----------------------------
df["attendance_flag"] = df["attendance_pct"].apply(lambda x: "Defaulter" if x < 75 else "Safe")
df["attendance_flag"].value_counts()

In [ ]:
"""Q16. Use pd.cut() to bin students into attendance ranges: [0-50, 50-75, 75-90, 90-100] and label them as ["Critical", "Warning", "Good", "Excellent"]."""
#-----------------------------Write Solution Code-----------------------------
# Bins are (0,50], (50,75], (75,90], (90,100]; include_lowest keeps 0% students
df["attendance_range"] = pd.cut(
    df["attendance_pct"],
    bins=[0, 50, 75, 90, 100],
    labels=["Critical", "Warning", "Good", "Excellent"],
    include_lowest=True,
)
display(df[["student_name", "attendance_pct", "attendance_range"]].head(10))
df["attendance_range"].value_counts().reindex(["Critical", "Warning", "Good", "Excellent"])

### Section 5: Advanced Analysis 

In [ ]:
"""Q17. Using pivot_table(), create a summary showing the count of students by Status and Attendance Category (created in Q10)."""
#-----------------------------Write Solution Code-----------------------------
pd.pivot_table(
    df,
    index="status",
    columns="attendance_category",
    values="roll_no",
    aggfunc="count",
    fill_value=0,
)[["Excellent", "Good", "Average", "Poor"]]

In [ ]:
"""Q18. Identify duplicate student names (e.g., "ANANYA", "DISHA") and display all rows where the base name repeats. Use string manipulation to extract the base name."""
#-----------------------------Write Solution Code-----------------------------
# Base name = name with the bracketed part (and the space / hyphen before it) removed
df["base_name"] = (
    df["student_name"]
      .str.replace(r"[\s\-]*\(.*?\)", "", regex=True)
      .str.strip()
)

dup_names = df[df["base_name"].duplicated(keep=False)].sort_values("base_name")
dup_names[["roll_no", "student_name", "base_name"]]

In [ ]:
""""Q19. Merge this dataset with a hypothetical Student_Details DataFrame containing Roll No, Branch, and Semester. Perform an inner join and display the merged result."""
#-----------------------------Write Solution Code-----------------------------
# Hypothetical Student_Details table (a few students left out on purpose,
# so the inner join visibly drops the non-matching rows)
student_details = pd.DataFrame({
    "roll_no": df["roll_no"].iloc[:-3],
    "branch": "CSE",
    "semester": 5,
})
student_details.loc[student_details["roll_no"].str.startswith("4MW25"), "semester"] = 3

merged = df.merge(student_details, on="roll_no", how="inner")
print("Rows before:", len(df), "| Student_Details:", len(student_details), "| After inner join:", len(merged))
merged[["roll_no", "student_name", "branch", "semester", "attendance_pct", "status"]].head(10)

In [ ]:
"""Q20. Export the following to separate CSV files:
(a) All "Critical" students
(b) Summary statistics grouped by Status
(c) Top 10 students by Risk Score"""
#-----------------------------Write Solution Code-----------------------------
# (a) All "Critical" students
df[df["status"] == "Critical"].to_csv("critical_students.csv", index=False)

# (b) Summary statistics grouped by Status
df.groupby("status")[["attendance_pct", "present", "absent", "consecutive_absences"]].describe().to_csv("summary_by_status.csv")

# (c) Top 10 students by Risk Score
df.nlargest(10, "risk_score").to_csv("top10_risk_score.csv", index=False)

print("Exported: critical_students.csv, summary_by_status.csv, top10_risk_score.csv")